# 00 - Starter Guide: Built-in Multi-Energy Model Catalog

Welcome to the **`energysim` model catalog starter guide**!

While `energysim` can co-simulate complex multi-domain systems with third-party solvers (pandapower, PyPSA, OpenDSS, FMUs, MATLAB/Octave, PowerFactory), it also comes equipped with an extensive library of **first-principles and empirical physical component models** out of the box.

In this starter tutorial, you will learn:
1. How `energysim` models are structured (`ModelComponent` base class).
2. How to inspect any model's declared inputs, outputs, states, and parameters.
3. How to test any model in **standalone mode** (stepping directly without a canvas).
4. Overview of the built-in catalog:
   - **Storage**: `BatteryBESS`, `FlowBattery`, `Flywheel`
   - **Renewables**: `PhysicsPV`, `WindTurbine`, `SolarThermalCollector`
   - **Hydrogen & Sector Coupling**: `PEMElectrolyzer`, `AlkalineElectrolyzer`, `H2Compressor`, `H2StorageTank`, `FuelCell_PEM`
   - **Thermal & HVAC**: `IndustrialHeatPump`, `ElectricBoiler`, `ThermalStorageTank`, `GasTurbine_OCGT`
   - **Flexible Loads**: `DataCenterLoad`, `IndustrialProcessSteam`
5. How to plug any of these models into an `energysim.world` co-simulation.


In [ ]:
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import energysim.models as models
from energysim import world

print("energysim models library imported successfully!")


## 1. Inspecting Built-in Archetypes and Models

`energysim` provides pre-configured industrial equipment archetypes (e.g. 2-hour 2 MW utility BESS, 5 MW PEM electrolyzer, utility solar farm).
You can inspect their default specifications with one line:


In [ ]:
# List key generic archetypes
for name, desc in list(models.archetypes.PROVENANCE.items())[:6]:
    print(f"- {name}: {desc}")


## 2. Standalone Model Testing (Zero Canvas Overhead)

Every `ModelComponent` in `energysim` can be stepped directly in pure Python without initializing a `world` canvas. This is ideal for unit tests, parameter calibration, or component-level sizing:


In [ ]:
# Instantiate a 2 MW / 4 MWh Li-ion Battery
bess = models.archetypes.BESS_2h_2MW(name="substation_battery")

print("Inputs :", bess.inputs)
print("Outputs:", bess.outputs)

# Step the battery standalone with a 1.5 MW charging command for 1 hour (3600 s)
bess.init()
bess.set_value(["P_cmd"], [1.5])
bess.step(3600.0)

soc, p_actual, t_batt = bess.get_value(["SoC", "P_actual", "T_batt"], 3600.0)
print(f"SoC after 1 hr charging at 1.5 MW: {soc:.3f}")
print(f"Terminal power delivered: {p_actual:.3f} MW")
print(f"Internal cell temperature: {t_batt:.1f} °C")


## 3. Storage Technologies: Lithium BESS vs Vanadium Flow vs Flywheel

`energysim` includes distinct storage physics:
- **`BatteryBESS`**: Li-ion with Equivalent Circuit Model (ECM), internal resistance, thermal heating, and optional degradation.
- **`FlowBattery`**: Vanadium redox flow battery (VRFB) with separate energy (tank electrolyte volume) and power (cell stack area) scaling, plus pump parasitic losses.
- **`Flywheel`**: High-rate kinetic energy storage with self-discharge standby loss, high round-trip efficiency, and unlimited cycling.


In [ ]:
flow_bat = models.FlowBattery(name="vanadium_flow", stack_p_max_mw=0.1, tank_capacity_mwh=0.5)
flywheel = models.Flywheel(name="freq_flywheel", capacity_kwh=25.0, p_max_kw=250.0)

print(f"Flow battery stack power: {flow_bat.stack_p_max_mw} MW, tank capacity: {flow_bat.tank_capacity_mwh} MWh")
print(f"Flywheel max power: {flywheel.p_max} kW, capacity: {flywheel.capacity_kwh} kWh")


## 4. Renewable Generation: Solar PV & Wind Turbines

- **`PhysicsPV`**: 5-parameter single-diode or empirical model with temperature coefficients and POA irradiance tracking.
- **`WindTurbine`**: Power curve modeling with cut-in, rated, and cut-out wind speed aerodynamics.


In [ ]:
wind = models.WindTurbine(name="turb_1", p_nom_kw=2000.0, v_cut_in=3.0, v_rated=12.0, v_cut_out=25.0)
wind.init()

# Evaluate wind power across wind speeds
speeds = np.linspace(0, 30, 100)
wind_p = []
for idx, v in enumerate(speeds):
    wind.set_value(["v_wind"], [v])
    wind.step((idx + 1) * 60.0)
    wind_p.append(wind.get_value(["P_actual"], (idx + 1) * 60.0)[0])

plt.figure(figsize=(7, 3.5))
plt.plot(speeds, wind_p, color="teal", lw=2)
plt.axvline(3.0, color="k", ls="--", lw=0.8, label="Cut-in (3 m/s)")
plt.axvline(12.0, color="orange", ls="--", lw=0.8, label="Rated (12 m/s)")
plt.axvline(25.0, color="red", ls="--", lw=0.8, label="Cut-out (25 m/s)")
plt.xlabel("Wind speed [m/s]")
plt.ylabel("Power output [kW]")
plt.title("Wind Turbine Aerodynamic Power Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Hydrogen & Sector Coupling

Green hydrogen hubs couple electric power to chemical energy:
- `PEMElectrolyzer` & `AlkalineElectrolyzer`: dynamic Faraday efficiency, polarization curve, hydrogen production rate (kg/h).
- `H2Compressor`: multi-stage compression energy requirements.
- `H2StorageTank`: pressurized buffer storage.
- `FuelCell_PEM`: reconversion of hydrogen back to electricity and heat.


In [ ]:
pem = models.PEMElectrolyzer(name="pem_stack", p_nominal_mw=1.0)
pem.init()
pem.set_value(["P_cmd"], [0.8])
pem.step(900.0)

m_dot_h2, spec_kwh_kg = pem.get_value(["m_dot_H2_kg_hr", "specific_consumption_kWh_kg"], 900.0)
print(f"PEM Electrolyzer at 0.8 MW input:")
print(f"  H2 Production Rate    : {m_dot_h2:.2f} kg/h")
print(f"  Specific Consumption  : {spec_kwh_kg:.1f} kWh/kg")


## 6. Integrating Catalog Models into a Co-Simulation World

Now let's connect multiple catalog models together in an `energysim.world` co-simulation with automatic unit conversion:


In [ ]:
sim = world(start_time=0, stop_time=3600*12, t_macro=900, power_unit="kW")

# Add components
sim.add_model("solar", models.PhysicsPV(name="solar", p_nom_kw=500.0))
sim.add_model("battery", models.BatteryBESS(name="battery", capacity_kwh=1000.0, p_max_kw=300.0, soc_init=0.4))
sim.add_model("heat_pump", models.IndustrialHeatPump(name="heat_pump", q_max_mw=0.4))

# Dynamic boundary conditions (signals)
sim.add_signal("irradiance", lambda t: max(0.0, 950.0 * np.sin(np.pi * t / (12*3600))), step_size=900, units={"y": "W/m2"})
sim.add_signal("heat_demand", 0.25, step_size=900, units={"y": "MW"})
sim.add_signal("bess_cmd", 150.0, step_size=900, units={"y": "kW"})

# Wire connections
sim.add_connections({
    "irradiance.y": "solar.G_poa",
    "bess_cmd.y": "battery.P_cmd",
    "heat_demand.y": "heat_pump.Q_heat_cmd_MW",
})

# Run simulation
res = sim.simulate(pbar=False)
df = res.to_frame()

print("Co-simulation complete! Variables recorded:", len(df.columns))
display(df[["solar.P_ac", "battery.SoC", "battery.P_actual", "heat_pump.P_elec_MW"]].head())


## Next Steps

Now that you have explored the built-in model library, check out the specialized tutorials:
- **01 Quickstart**: Detailed multi-energy time-synchronization concepts.
- **02 Grid Solvers**: Co-simulating pandapower, PyPSA, and OpenDSS power flows.
- **03 Custom Models**: Authoring your own custom physical equations with `ModelComponent`.
- **07 Green Hydrogen**: Comprehensive multi-stage hydrogen valley co-simulation.
- **20 Simplified Modeling**: Fast vectorized fleets and Monte Carlo ensembles.
